# AIDAO 2025 — occupancy grid in bird's-eye view

Final-round solution: predict a static occupancy grid around a self-driving car
from four onboard cameras. Lift-Splat-Shoot implemented from scratch in PyTorch.

**Metric:** mean IoU over `{free, occupied}` with an ignore label.
**Result:** IoU 0.5606 on the private test set.

Pipeline: image features -> per-pixel depth distribution -> unproject into the car
frame using the calibration *of this very frame* -> scatter into a BEV grid ->
fuse the four cameras -> decode occupancy logits.

## Dataset schema

In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import matplotlib.pyplot as plt
from PIL import Image
from torch.amp import autocast, GradScaler
from torch.utils.data import ConcatDataset, DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

# Columns of info.csv. Every frame carries four views plus their calibration.
CAMERA_NAMES = [
    "/camera/inner/frontal/middle",
    "/camera/inner/frontal/far",
    "/side/left/forward",
    "/side/right/forward",
]
INTRINSICS_NAMES = [f"{cam}/intrinsic_params" for cam in CAMERA_NAMES]
CAR2CAM_NAMES = [f"{cam}/car_to_cam" for cam in CAMERA_NAMES]
GRID_NAME = "gt_occupancy_grid"

DATA_ROOT = Path("./")
TRAIN_DIR = DATA_ROOT / "train"
VAL_DIR = DATA_ROOT / "val"
TEST_DIR = DATA_ROOT / "test"

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

## Dataset

Each sample returns the four images, their intrinsics and `car_to_cam` matrices, and
the **original** image resolutions. The last one matters: images are resized before the
backbone, but the intrinsics describe the original frame, so the pixel grid has to be
mapped back before it is unprojected. The four cameras do not even share a resolution.

Ground truth uses `255` for "unknown"; it is remapped to `-1` and masked out of the loss.

In [ ]:
class AutonomyDataset(Dataset):
    """Frames of four calibrated cameras with an optional BEV occupancy target."""

    def __init__(self, root_dir: Path, mode: str = "train", img_size: tuple[int, int] = (256, 512)):
        assert mode in ("train", "val", "test")
        self.root_dir = Path(root_dir)
        self.mode = mode
        self.img_size = img_size

        csv_path = self.root_dir / "info.csv"
        if not csv_path.exists():
            raise FileNotFoundError(f"info.csv not found at {csv_path}")
        self.info = pd.read_csv(csv_path, index_col=0)

        self.img_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize(img_size),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ])

    def __len__(self) -> int:
        return len(self.info)

    def _resolve_path(self, raw: str) -> Path:
        """info.csv paths come in several flavours; try them in order of specificity."""
        path = Path(str(raw))
        if path.exists():
            return path

        relative = self.root_dir / str(raw).lstrip("./")
        if relative.exists():
            return relative

        if len(path.parts) > 1:
            stripped = self.root_dir.joinpath(*path.parts[1:])
            if stripped.exists():
                return stripped

        return relative

    def _load_cameras(self, row: pd.Series) -> tuple[torch.Tensor, torch.Tensor]:
        images, original_hw = [], []
        for cam in CAMERA_NAMES:
            with Image.open(self._resolve_path(row[cam])) as raw:
                image = raw.convert("RGB")
                width, height = image.size
                original_hw.append([height, width])
                images.append(self.img_transform(image))
        return torch.stack(images), torch.tensor(original_hw, dtype=torch.float32)

    def _load_calibration(self, row: pd.Series) -> tuple[torch.Tensor, torch.Tensor]:
        intrinsics, car2cams = [], []
        for intr_col, c2c_col in zip(INTRINSICS_NAMES, CAR2CAM_NAMES):
            intrinsics.append(torch.from_numpy(np.load(self._resolve_path(row[intr_col]))).float())
            car2cams.append(torch.from_numpy(np.load(self._resolve_path(row[c2c_col]))).float())
        return torch.stack(intrinsics), torch.stack(car2cams)

    def _load_grid(self, row: pd.Series) -> torch.Tensor:
        grid = np.squeeze(np.load(self._resolve_path(row[GRID_NAME]))).astype(np.int16)
        grid[grid == 255] = -1  # unknown cells are ignored by loss and metric
        tensor = torch.from_numpy(grid)
        return tensor.unsqueeze(0) if tensor.ndim == 2 else tensor

    def __getitem__(self, idx: int) -> dict[str, Any]:
        row = self.info.iloc[idx]
        images, original_hw = self._load_cameras(row)
        intrinsics, car2cams = self._load_calibration(row)

        sample = {
            "images": images,            # cams x 3 x H x W
            "intrinsics": intrinsics,    # cams x 3 x 4
            "car2cams": car2cams,        # cams x 4 x 4
            "orig_hw": original_hw,      # cams x 2, needed to rescale the intrinsics
            "idx": idx,
        }
        if self.mode != "test":
            sample["grid"] = self._load_grid(row)
        return sample


def photometric_augment_gpu(images: torch.Tensor) -> torch.Tensor:
    """Brightness / contrast / noise jitter, shared across the cameras of one sample."""
    images = images if images.is_floating_point() else images.float()
    device = images.device
    batch = images.shape[0]

    if torch.rand(1, device=device) < 0.8:
        brightness = 0.1 * (2 * torch.rand(batch, 1, 1, 1, 1, device=device) - 1.0)
        images = images + brightness

    if torch.rand(1, device=device) < 0.8:
        contrast = 1.0 + 0.1 * (2 * torch.rand(batch, 1, 1, 1, 1, device=device) - 1.0)
        images = images * contrast

    if torch.rand(1, device=device) < 0.5:
        images = images + torch.randn_like(images) * 0.05

    return images

## Geometry

The BEV grid covers 150 m ahead and 100 m across in 188x126 cells (~0.8 m per cell).
Depth is discretised into 64 bins between 1 m and 120 m; only points whose height falls
inside `[Z_MIN, Z_MAX]` are scattered, which drops the sky and the road surface.

In [ ]:
@dataclass
class LSSConfig:
    H_img: int = 320
    W_img: int = 640
    downsample: int = 8  # backbone stride -> 40 x 80 feature map

    X_MIN: float = 0.0
    X_MAX: float = 150.0
    Y_MIN: float = -50.0
    Y_MAX: float = 50.0
    H_bev: int = 188
    W_bev: int = 126

    D: int = 64
    DEPTH_MIN: float = 1.0
    DEPTH_MAX: float = 120.0
    use_log_depth: bool = False

    Z_MIN: float = -1.0
    Z_MAX: float = 3.0

    @property
    def RES_X(self) -> float:
        return (self.X_MAX - self.X_MIN) / self.H_bev

    @property
    def RES_Y(self) -> float:
        return (self.Y_MAX - self.Y_MIN) / self.W_bev


def extract_K_from_intrinsics(intrinsics: torch.Tensor) -> torch.Tensor:
    """Accept both 3x4 and 3x3 intrinsics, return the 3x3 block."""
    if intrinsics.shape[-1] == 4:
        return intrinsics[..., :3]
    if intrinsics.shape[-1] == 3:
        return intrinsics
    raise ValueError(f"unexpected intrinsics shape: {intrinsics.shape}")


def get_cam_orig_sizes(dataset: AutonomyDataset) -> list[tuple[int, int]]:
    """Original (H, W) per camera, read once from the first row of info.csv."""
    row = dataset.info.iloc[0]
    sizes = []
    for cam in CAMERA_NAMES:
        with Image.open(dataset._resolve_path(row[cam])) as image:
            width, height = image.size
        sizes.append((height, width))
    return sizes

## Model

`ResNet50Backbone` is shared by all four cameras and merges strides /8 and /16 in an
FPN fashion, so the features keep small far-away objects without paying for a /4 map.

`DepthHead` predicts a categorical depth distribution per feature pixel. Its softmax
temperature is learned (clamped to `[0.3, 3.0]`) — early on the model is happier
spreading mass across bins, later it sharpens on its own.

`LSSOccupancyModel.forward` recomputes the projection **for every frame** from that
frame's intrinsics and `car_to_cam`, instead of precomputing one fixed geometry for the
whole dataset. This was the single biggest source of improvement.

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-excitation over BEV channels."""

    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.gate = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.gate(self.pool(x))


class ResidualBlock2D(nn.Module):
    """Conv-BN-ReLU-Conv-BN with a skip connection and optional dilation."""

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        stride: int = 1,
        dilation: int = 1,
        norm_layer=nn.BatchNorm2d,
    ):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, kernel_size=3, stride=stride,
            padding=dilation, dilation=dilation, bias=False,
        )
        self.bn1 = norm_layer(out_channels)
        self.conv2 = nn.Conv2d(
            out_channels, out_channels, kernel_size=3, stride=1,
            padding=dilation, dilation=dilation, bias=False,
        )
        self.bn2 = norm_layer(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.downsample = None
        if in_channels != out_channels or stride != 1:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                norm_layer(out_channels),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x if self.downsample is None else self.downsample(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)


class ResNet50Backbone(nn.Module):
    """ImageNet ResNet-50 truncated at layer3, with an FPN merge of /8 and /16."""

    def __init__(self, out_channels: int = 64, weights_path: str | None = None):
        super().__init__()
        if weights_path is not None:
            base = models.resnet50(weights=None)
            base.load_state_dict(torch.load(weights_path, map_location="cpu"))
        else:
            base = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        self.stem = nn.Sequential(base.conv1, base.bn1, base.relu, base.maxpool)
        self.layer1 = base.layer1
        self.layer2 = base.layer2
        self.layer3 = base.layer3

        self.lateral_8 = nn.Conv2d(512, out_channels, kernel_size=1)
        self.lateral_16 = nn.Conv2d(1024, out_channels, kernel_size=1)
        self.out_channels = out_channels

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.layer1(self.stem(x))
        feat_8 = self.layer2(x)
        feat_16 = self.layer3(feat_8)

        merged = self.lateral_8(feat_8) + F.interpolate(
            self.lateral_16(feat_16),
            size=feat_8.shape[-2:],
            mode="bilinear",
            align_corners=False,
        )
        return merged


class DepthHead(nn.Module):
    """Categorical depth distribution per feature pixel, with a learned temperature."""

    def __init__(self, feat_channels: int, num_bins: int):
        super().__init__()
        self.conv1 = nn.Conv2d(feat_channels, feat_channels, kernel_size=3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(feat_channels)
        self.conv2 = nn.Conv2d(feat_channels, feat_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(feat_channels)
        self.relu = nn.ReLU(inplace=True)
        self.proj = nn.Conv2d(feat_channels, num_bins, kernel_size=1)
        self.log_temp = nn.Parameter(torch.zeros(1))

    def forward(self, feat: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        out = self.relu(self.bn1(self.conv1(feat)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + feat)

        logits = self.proj(out)
        temperature = torch.exp(self.log_temp).clamp(0.3, 3.0)
        return torch.softmax(logits / temperature, dim=1), logits


class LSSOccupancyModel(nn.Module):
    """Lift-Splat-Shoot with per-frame calibration.

    Shared backbone over the cameras -> depth distribution -> scatter into BEV using
    this frame's calibration -> max/mean fusion across cameras -> dilated decoder.
    """

    def __init__(self, cfg: LSSConfig, cam_orig_sizes: list[tuple[int, int]], feat_channels: int = 64):
        super().__init__()
        self.cfg = cfg
        self.n_cams = len(cam_orig_sizes)

        self.backbone = ResNet50Backbone(
            out_channels=feat_channels,
            weights_path="./resnet50_imagenet1k_default.pth",  # None -> download from torchvision
        )
        self.depth_head = DepthHead(feat_channels, cfg.D)

        h_feat = cfg.H_img // cfg.downsample
        w_feat = cfg.W_img // cfg.downsample
        rows, cols = torch.meshgrid(
            torch.arange(h_feat, dtype=torch.float32),
            torch.arange(w_feat, dtype=torch.float32),
            indexing="ij",
        )
        # Pixel centres of the feature map, in resized-image coordinates.
        self.register_buffer("u_resized_flat", ((cols + 0.5) * cfg.downsample).view(1, -1))
        self.register_buffer("v_resized_flat", ((rows + 0.5) * cfg.downsample).view(1, -1))

        self.bev_fusion = nn.Sequential(
            nn.Conv2d(feat_channels * 2, feat_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(feat_channels),
            nn.ReLU(inplace=True),
        )
        self.bev_se = SEBlock(feat_channels)
        self.bev_context = nn.Sequential(
            ResidualBlock2D(feat_channels, 128, dilation=2),
            ResidualBlock2D(128, feat_channels, dilation=4),
        )
        self.bev_decoder = nn.Sequential(
            ResidualBlock2D(feat_channels, 128),
            ResidualBlock2D(128, 128),
            nn.Dropout2d(p=0.1),
            ResidualBlock2D(128, 64),
            nn.Conv2d(64, 1, kernel_size=1),
        )

    def _compute_bev_indices_for_cam(
        self,
        intrinsics: torch.Tensor,
        car2cam: torch.Tensor,
        orig_hw: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Map every (feature pixel, depth bin) pair to a flat BEV cell index.

        Invalid pairs — outside the grid or outside the height band — get index -1.
        """
        cfg = self.cfg
        device = intrinsics.device
        batch = intrinsics.shape[0]

        k_inv = torch.inverse(extract_K_from_intrinsics(intrinsics))

        # The intrinsics belong to the original image, the feature grid to the resized
        # one, so undo the resize before casting rays.
        orig_h = orig_hw[:, 0:1]
        orig_w = orig_hw[:, 1:2]
        scale_x = cfg.W_img / torch.clamp(orig_w, min=1.0)
        scale_y = cfg.H_img / torch.clamp(orig_h, min=1.0)

        u = self.u_resized_flat.to(device).expand(batch, -1) / scale_x
        v = self.v_resized_flat.to(device).expand(batch, -1) / scale_y
        pixels = torch.stack([u, v, torch.ones_like(u)], dim=1)  # B x 3 x N
        rays = torch.bmm(k_inv, pixels)

        if cfg.use_log_depth:
            depths = torch.logspace(
                math.log10(cfg.DEPTH_MIN), math.log10(cfg.DEPTH_MAX), cfg.D, device=device
            )
        else:
            depths = torch.linspace(cfg.DEPTH_MIN, cfg.DEPTH_MAX, cfg.D, device=device)

        num_pixels = pixels.shape[-1]
        points_cam = rays.view(batch, 3, 1, num_pixels) * depths.view(1, cfg.D, 1)
        points_cam = torch.cat(
            [points_cam, torch.ones((batch, 1, cfg.D, num_pixels), device=device)], dim=1
        )

        cam2car = torch.inverse(car2cam.to(device))
        points_car = torch.bmm(cam2car, points_cam.view(batch, 4, -1)).view(batch, 4, cfg.D, num_pixels)
        x_car, y_car, z_car = points_car[:, 0], points_car[:, 1], points_car[:, 2]

        col = torch.floor((x_car - cfg.X_MIN) / cfg.RES_X).long()
        row = torch.floor((y_car - cfg.Y_MIN) / cfg.RES_Y).long()
        valid = (
            (z_car >= cfg.Z_MIN) & (z_car <= cfg.Z_MAX)
            & (col >= 0) & (col < cfg.H_bev)
            & (row >= 0) & (row < cfg.W_bev)
        )

        bev_index = col * cfg.W_bev + row
        bev_index[~valid] = -1

        h_feat = cfg.H_img // cfg.downsample
        w_feat = cfg.W_img // cfg.downsample
        shape = (batch, cfg.D, h_feat, w_feat)
        return (
            bev_index.view(shape).permute(0, 2, 3, 1).contiguous(),
            valid.view(shape).permute(0, 2, 3, 1).contiguous(),
        )

    def lift_splat_single_cam(
        self,
        feat: torch.Tensor,
        depth_prob: torch.Tensor,
        bev_indices: torch.Tensor,
        valid_mask: torch.Tensor,
    ) -> torch.Tensor:
        """Weight features by depth probability and sum them into their BEV cells."""
        batch, channels, h_feat, w_feat = feat.shape
        num_bins = depth_prob.shape[1]
        num_cells = self.cfg.H_bev * self.cfg.W_bev

        weighted = feat.view(batch, channels, h_feat * w_feat, 1).expand(-1, -1, -1, num_bins)
        weighted = (weighted * depth_prob.view(batch, 1, h_feat * w_feat, num_bins))
        weighted = weighted.view(batch, channels, -1)

        flat_indices = bev_indices.view(batch, -1)
        flat_valid = valid_mask.view(batch, -1)
        bev = weighted.new_zeros((batch, channels, num_cells))

        for b in range(batch):
            selected = flat_valid[b].nonzero(as_tuple=False).squeeze(1)
            if selected.numel() == 0:
                continue
            cells = flat_indices[b, selected].clamp(0, num_cells - 1)
            bev[b].index_add_(1, cells, weighted[b, :, selected])

        return bev.view(batch, channels, self.cfg.H_bev, self.cfg.W_bev)

    def forward(
        self,
        images: torch.Tensor,
        intrinsics: torch.Tensor,
        car2cams: torch.Tensor,
        orig_hw: torch.Tensor,
    ) -> torch.Tensor:
        batch, n_cams = images.shape[:2]
        assert n_cams == self.n_cams, f"expected {self.n_cams} cameras, got {n_cams}"

        bev_per_cam = []
        for cam in range(n_cams):
            feat = self.backbone(images[:, cam])
            depth_prob, _ = self.depth_head(feat)
            bev_indices, valid_mask = self._compute_bev_indices_for_cam(
                intrinsics[:, cam], car2cams[:, cam], orig_hw[:, cam]
            )
            bev_per_cam.append(self.lift_splat_single_cam(feat, depth_prob, bev_indices, valid_mask))

        stacked = torch.stack(bev_per_cam, dim=1)
        # Max keeps the most confident camera, mean keeps agreement between them.
        fused = self.bev_fusion(torch.cat([stacked.max(dim=1).values, stacked.mean(dim=1)], dim=1))
        fused = self.bev_se(fused)
        return self.bev_decoder(self.bev_context(fused))

## Metric and loss

The score is a mean IoU over the two classes with unknown cells ignored, so the loss
optimises exactly that: weighted BCE plus a differentiable soft-IoU surrogate. The
`pos_weight` comes from the measured class balance rather than a guess.

In [ ]:
def compute_batch_mean_iou(
    logits: torch.Tensor, grid: torch.Tensor, threshold: float = 0.5
) -> torch.Tensor:
    """Mean IoU over {free, occupied}; cells labelled -1 are ignored."""
    with torch.no_grad():
        valid = grid != -1
        if valid.sum() == 0:
            return torch.tensor(0.0, device=logits.device)

        preds = (torch.sigmoid(logits) > threshold).long()[valid]
        target = (grid == 1).long()[valid]

        ious = []
        for cls in (1, 0):
            tp = ((preds == cls) & (target == cls)).sum()
            fp = ((preds == cls) & (target != cls)).sum()
            fn = ((preds != cls) & (target == cls)).sum()
            denom = tp + fp + fn
            if denom > 0:
                ious.append(tp.float() / denom.float())

        if not ious:
            return torch.tensor(0.0, device=logits.device)
        return sum(ious) / len(ious)


def soft_miou_loss(logits: torch.Tensor, grid: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """Differentiable stand-in for the mean IoU, masked to the labelled cells."""
    valid = grid != -1
    if valid.sum() == 0:
        return logits.new_tensor(0.0)

    probs = torch.sigmoid(logits).squeeze(1)
    target = (grid == 1).float().squeeze(1)
    mask = valid.float().squeeze(1)

    inter_pos = (mask * probs * target).sum()
    union_pos = (mask * (probs + target - probs * target)).sum() + eps

    neg_probs, neg_target = 1.0 - probs, 1.0 - target
    inter_neg = (mask * neg_probs * neg_target).sum()
    union_neg = (mask * (neg_probs + neg_target - neg_probs * neg_target)).sum() + eps

    return 1.0 - 0.5 * (inter_pos / union_pos + inter_neg / union_neg)


def estimate_class_balance(loader: DataLoader, max_batches: int = 50) -> tuple[int, int]:
    """Count labelled occupied / free cells to derive pos_weight."""
    positive = negative = 0
    for i, batch in enumerate(loader):
        grid = batch["grid"]
        valid = grid != -1
        positive += ((grid == 1) & valid).sum().item()
        negative += ((grid == 0) & valid).sum().item()
        if i + 1 >= max_batches:
            break
    return positive, negative

## Training

The backbone is pretrained, the heads are not, so they get separate learning rates
(1e-4 / 5e-4) under one cosine schedule. Mixed precision roughly halves the step time,
which is what makes ten epochs fit into the contest window.

The final run trains on `train + val`; the val split stays in the loop only as a
progress signal, so the numbers printed below are *not* a clean validation score.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

cfg = LSSConfig()
train_dataset = AutonomyDataset(TRAIN_DIR, mode="train", img_size=(cfg.H_img, cfg.W_img))
val_dataset = AutonomyDataset(VAL_DIR, mode="val", img_size=(cfg.H_img, cfg.W_img))
trainval_dataset = ConcatDataset([train_dataset, val_dataset])
print("train:", len(train_dataset), "val:", len(val_dataset), "train+val:", len(trainval_dataset))

cam_orig_sizes = get_cam_orig_sizes(train_dataset)
print("camera sizes:", cam_orig_sizes)

BATCH_SIZE = 8
NUM_WORKERS = 8
NUM_EPOCHS = 10
BEST_MODEL_PATH = "lss_occupancy_final.pth"

balance_loader = DataLoader(trainval_dataset, batch_size=4, shuffle=True, num_workers=4, pin_memory=True)
pos_count, neg_count = estimate_class_balance(balance_loader, max_batches=50)
pos_weight = torch.tensor(neg_count / max(pos_count, 1), device=device)
print("positive:", pos_count, "negative:", neg_count, "pos_weight:", pos_weight.item())

train_loader = DataLoader(
    trainval_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

model = LSSOccupancyModel(cfg, cam_orig_sizes, feat_channels=64).to(device)

backbone_params = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("backbone.")]
head_params = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("backbone.")]
optimizer = torch.optim.AdamW([
    {"params": backbone_params, "lr": 1e-4, "weight_decay": 1e-4},
    {"params": head_params, "lr": 5e-4, "weight_decay": 1e-4},
])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS * 2)
scaler = GradScaler("cuda")


def run_batch(batch: dict[str, torch.Tensor], train: bool) -> tuple[float, float] | None:
    images = batch["images"].to(device, non_blocking=True)
    intrinsics = batch["intrinsics"].to(device, non_blocking=True)
    car2cams = batch["car2cams"].to(device, non_blocking=True)
    orig_hw = batch["orig_hw"].to(device, non_blocking=True)
    grid = batch["grid"].to(device, non_blocking=True)

    valid = grid != -1
    if valid.sum() == 0:
        return None

    if train:
        images = photometric_augment_gpu(images)

    with autocast("cuda"):
        logits = model(images, intrinsics=intrinsics, car2cams=car2cams, orig_hw=orig_hw)
        bce = F.binary_cross_entropy_with_logits(
            logits[valid], (grid == 1).float()[valid], pos_weight=pos_weight
        )
        loss = bce + 0.3 * soft_miou_loss(logits, grid)

    if train:
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    return loss.item(), compute_batch_mean_iou(logits.float(), grid).item()


best_val_iou = 0.0
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    train_stats = [
        result for batch in tqdm(train_loader, desc=f"epoch {epoch:02d} train", leave=False)
        if (result := run_batch(batch, train=True)) is not None
    ]

    model.eval()
    with torch.no_grad():
        val_stats = [
            result for batch in tqdm(val_loader, desc=f"epoch {epoch:02d} val", leave=False)
            if (result := run_batch(batch, train=False)) is not None
        ]

    train_loss, train_iou = (sum(x) / len(train_stats) for x in zip(*train_stats))
    val_loss, val_iou = (sum(x) / len(val_stats) for x in zip(*val_stats))
    print(
        f"Epoch {epoch:02d}: train_loss={train_loss:.4f}, train_IoU={train_iou:.4f}, "
        f"val_loss={val_loss:.4f}, val_IoU={val_iou:.4f}"
    )

    if val_iou > best_val_iou:
        best_val_iou = val_iou
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        print(f"  -> saved {BEST_MODEL_PATH} (val IoU {best_val_iou:.4f})")

    scheduler.step()

## Threshold calibration

The decision threshold is a free parameter the loss never sees. Sweeping it moves the
balance between the two class IoUs; 0.55 came out best, and the curve is flat enough
around it that the choice is not fitting noise.

In [ ]:
model.load_state_dict(torch.load(BEST_MODEL_PATH, map_location=device))
model.to(device)
model.eval()


def evaluate_threshold(threshold: float) -> tuple[float, float, float, float]:
    """Return (mIoU, IoU_free, IoU_occupied, predicted occupied fraction)."""
    counts = {cls: {"tp": 0, "fp": 0, "fn": 0} for cls in (0, 1)}
    total_valid = total_positive = 0

    with torch.no_grad():
        for batch in val_loader:
            logits = model(
                batch["images"].to(device),
                intrinsics=batch["intrinsics"].to(device),
                car2cams=batch["car2cams"].to(device),
                orig_hw=batch["orig_hw"].to(device),
            )
            grid = batch["grid"].to(device)
            valid = grid != -1
            if valid.sum() == 0:
                continue

            preds = (torch.sigmoid(logits) > threshold).long()[valid]
            target = (grid == 1).long()[valid]
            total_valid += valid.sum().item()
            total_positive += (preds == 1).sum().item()

            for cls in (0, 1):
                counts[cls]["tp"] += ((preds == cls) & (target == cls)).sum().item()
                counts[cls]["fp"] += ((preds == cls) & (target != cls)).sum().item()
                counts[cls]["fn"] += ((preds != cls) & (target == cls)).sum().item()

    def iou(cls: int) -> float:
        c = counts[cls]
        denom = c["tp"] + c["fp"] + c["fn"]
        return 0.0 if denom == 0 else c["tp"] / denom

    iou_free, iou_occupied = iou(0), iou(1)
    return 0.5 * (iou_free + iou_occupied), iou_free, iou_occupied, total_positive / max(total_valid, 1)


best_threshold, best_miou = 0.5, -1.0
for threshold in np.linspace(0.35, 0.65, 7):
    miou, iou_free, iou_occupied, frac_positive = evaluate_threshold(float(threshold))
    print(
        f"thr={threshold:.2f}: mIoU={miou:.4f}, IoU_free={iou_free:.4f}, "
        f"IoU_occupied={iou_occupied:.4f}, frac_positive={frac_positive:.4f}"
    )
    if miou > best_miou:
        best_threshold, best_miou = float(threshold), miou

print(f"\nbest threshold={best_threshold:.3f}, mIoU={best_miou:.4f}")

## Qualitative check

In [ ]:
def visualize_val_sample(idx: int, threshold: float = 0.5) -> None:
    """Four camera views next to ground truth, prediction and raw probabilities."""
    sample = val_dataset[idx]
    with torch.no_grad():
        logits = model(
            sample["images"].unsqueeze(0).to(device),
            intrinsics=sample["intrinsics"].unsqueeze(0).to(device),
            car2cams=sample["car2cams"].unsqueeze(0).to(device),
            orig_hw=sample["orig_hw"].unsqueeze(0).to(device),
        )
    probs = torch.sigmoid(logits)[0, 0].cpu().numpy()
    prediction = (probs > threshold).astype(np.int16)

    ground_truth = sample["grid"].squeeze(0).cpu().numpy().copy()
    ground_truth[ground_truth == 255] = -1

    row = val_dataset.info.iloc[idx]
    extent = [-50, 50, 0, 150]

    figure = plt.figure(figsize=(24, 10))
    grid_spec = figure.add_gridspec(2, 4, width_ratios=[1, 1, 1.4, 1.4])

    for i, (r, c) in enumerate([(0, 0), (0, 1), (1, 0), (1, 1)]):
        with Image.open(val_dataset._resolve_path(row[CAMERA_NAMES[i]])) as image:
            axis = figure.add_subplot(grid_spec[r, c])
            axis.imshow(np.asarray(image.convert("RGB")))
            axis.set_title(CAMERA_NAMES[i], fontsize=9)
            axis.axis("off")

    for position, (data, title, cmap, limits) in enumerate([
        (ground_truth, f"ground truth (idx={idx})", "RdYlBu_r", (-1, 1)),
        (prediction, f"prediction (thr={threshold})", "RdYlBu_r", (-1, 1)),
    ]):
        axis = figure.add_subplot(grid_spec[position, 2])
        image = axis.imshow(data, cmap=cmap, extent=extent, origin="lower", vmin=limits[0], vmax=limits[1])
        axis.set_title(title)
        axis.invert_xaxis()
        axis.grid(True, alpha=0.3)
        plt.colorbar(image, ax=axis, shrink=0.8)

    axis = figure.add_subplot(grid_spec[:, 3])
    image = axis.imshow(probs, cmap="viridis", extent=extent, origin="lower")
    axis.set_title("sigmoid(logits)")
    axis.invert_xaxis()
    axis.grid(True, alpha=0.3)
    plt.colorbar(image, ax=axis, shrink=0.8)

    plt.tight_layout()
    plt.show()


for sample_idx in (0, 10, 50):
    visualize_val_sample(sample_idx, threshold=best_threshold)

## Submission

In [ ]:
import shutil

SUB_DIR = Path("./sub")
GRID_SUBDIR = SUB_DIR / "predicted_static_grids"

shutil.rmtree(SUB_DIR, ignore_errors=True)
GRID_SUBDIR.mkdir(parents=True, exist_ok=True)

test_info = pd.read_csv(TEST_DIR / "info.csv", index_col=0)
test_dataset = AutonomyDataset(TEST_DIR, mode="test", img_size=(cfg.H_img, cfg.W_img))
# batch_size=1 keeps row i of info.csv aligned with prediction i
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4, pin_memory=True)

model.eval()
with torch.inference_mode():
    for i, batch in enumerate(tqdm(test_loader)):
        logits = model(
            batch["images"].to(device),
            batch["intrinsics"].to(device),
            batch["car2cams"].to(device),
            batch["orig_hw"].to(device),
        )
        prediction = (torch.sigmoid(logits) > best_threshold).float()
        grid = prediction[0, 0].cpu().numpy().astype(np.int32)[None, ...]  # 1 x 188 x 126

        target_name = test_info.iloc[i]["predicted_occupancy_grid"].split("/")[-1]
        np.save(GRID_SUBDIR / target_name, grid)

shutil.copy(TEST_DIR / "info.csv", SUB_DIR / "info.csv")
shutil.make_archive("submission", "zip", SUB_DIR)

saved = list(GRID_SUBDIR.glob("*.npy"))
print(f"rows in info.csv: {len(test_info)}, saved grids: {len(saved)}")
print("sample grid:", np.load(saved[0]).shape, np.load(saved[0]).dtype)